# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Data + the label: `declined_30d_future`

The baseline queue is one row per content item at a single **decision point `D`**. Every feature window ends at `D`; the label is a *future* outcome measured in the 30 days after `D`, so it was not knowable at decision time.

- **Table:** `fact_content_daily_performance` (warehouse). Iterating on the two month partitions that cover the window (`month=2026-05` = prior, `month=2026-06` = label). The `_sample` table is June-only, so it cannot feed the prior window on its own.
- **Decision point:** `D = 2026-05-31`. Facts run through `2026-06-30` (freshest 3 days cut), so the 30-day label window `(D, D+30]` is fully observed.
- **Label:** `declined_30d_future` = 1 when GSC impressions in `(D, D+30]` are under 80% of impressions in `(D-30, D]`, else 0. Pages without 30 days of history or with too few observed days are **not labelable** and get `NaN` — they are not counted as "not declined".
- This is the label the w05 model will predict; the baseline and the model share this slice.

In [1]:
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import numpy as np
import pandas as pd

load_dotenv()  # repo .env carries HF_TOKEN locally; Colab falls back to the prompt below.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Iterate on the two partitions that cover the feature + label window (D-30 .. D+30).
# Final pass: FACT = TABLES["fact_daily"] once, then cache to work/outputs/.
FACT = (
    "read_parquet(['"
    + f"{REL}/fact_content_daily_performance/month=2026-05/*.parquet',"
    + f"'{REL}/fact_content_daily_performance/month=2026-06/*.parquet'])"
)
print("FACT =", FACT)

FACT = read_parquet(['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet','hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet'])


In [2]:
# Skill verification: COUNT(*) + MIN/MAX(report_date) on the table I iterate on.
row = con.sql(f"SELECT COUNT(*) AS n, MIN(report_date), MAX(report_date) FROM {FACT}").fetchone()
print(f"FACT: {row[0]:,} rows | {row[1]} -> {row[2]}  (May + Jun 2026 partitions)")

n_full = con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily']}").fetchone()[0]
print(f"full fact table: {n_full:,} rows (skill expects 78,835,655)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FACT: 23,381,448 rows | 2026-05-01 -> 2026-06-30  (May + Jun 2026 partitions)


full fact table: 78,835,655 rows (skill expects 78,835,655)


In [3]:
# Grain probe: client x content x date should be one row. w03 found 6,390 dups in the full table.
dups = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT client_hash_id, content_hash_id, report_date
        FROM {FACT}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f"duplicate rows at the grain: {dups:,}")

# Panel coverage: the unbalanced panel means per-client windows, never one global calendar window.
clients = con.sql(f"""
    SELECT c.client_hash_id, c.has_gsc_access, c.gsc_data_start,
           MAX(f.report_date) AS last_report
    FROM {TABLES['dim_clients']} c
    LEFT JOIN {TABLES['fact_daily']} f USING (client_hash_id)
    GROUP BY 1, 2, 3
""").df()
with_gsc = clients[clients["has_gsc_access"]]
print(f"clients: {len(clients)} | with GSC access: {len(with_gsc)}")
print("gsc_data_start range:", with_gsc["gsc_data_start"].min(), "->", with_gsc["gsc_data_start"].max())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate rows at the grain: 6,390


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

clients: 104 | with GSC access: 67
gsc_data_start range: 2025-01-27 00:00:00 -> 2026-06-02 00:00:00


In [4]:
END_DATE = pd.Timestamp(con.sql(f"SELECT MAX(report_date) FROM {FACT}").fetchone()[0])
D = END_DATE - pd.Timedelta(days=30)
print("facts end:", END_DATE.date())
print("decision point D:", D.date())
print("prior window (features):", (D - pd.Timedelta(days=30)).date(), "->", D.date())
print("label window (future):", (D + pd.Timedelta(days=1)).date(), "->", END_DATE.date())
print("label fully observed because D + 30 == END")

facts end: 2026-06-30
decision point D: 2026-05-31
prior window (features): 2026-05-01 -> 2026-05-31
label window (future): 2026-06-01 -> 2026-06-30
label fully observed because D + 30 == END


#### Build the label

The windowed SQL does everything in one pass over the daily facts: per-content prior-30d and future-30d GSC impressions, observed-day counts, and client/content coverage filters. The pandas step applies the floors and derives the binary label.

In [5]:
SQL = f"""
WITH dedup AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY client_hash_id, content_hash_id, report_date) AS rn
    FROM {FACT}
),
ok AS (SELECT * FROM dedup WHERE rn = 1),
client_bounds AS (
    SELECT c.client_hash_id, c.gsc_data_start, MAX(f.report_date) AS last_report
    FROM {TABLES['dim_clients']} c
    LEFT JOIN {TABLES['fact_daily']} f USING (client_hash_id)
    GROUP BY 1, 2
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(CASE WHEN f.gsc_data_available
                 AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                 THEN f.gsc_impressions ELSE 0 END) AS prior_imp,
        COUNT(CASE WHEN f.gsc_data_available
                   AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                   THEN 1 END) AS prior_obs_days,
        SUM(CASE WHEN f.gsc_data_available
                 AND f.report_date > DATE '{D.date()}' AND f.report_date <= DATE '{D.date()}' + INTERVAL 30 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS future_imp,
        COUNT(CASE WHEN f.gsc_data_available
                   AND f.report_date > DATE '{D.date()}' AND f.report_date <= DATE '{D.date()}' + INTERVAL 30 DAY
                   THEN 1 END) AS future_obs_days
    FROM ok f
    JOIN client_bounds b USING (client_hash_id)
    LEFT JOIN {TABLES['dim_content']} c USING (client_hash_id, content_hash_id)
    WHERE b.gsc_data_start <= DATE '{D.date()}' - INTERVAL 30 DAY
      AND b.last_report >= DATE '{D.date()}' + INTERVAL 30 DAY
      AND c.content_created_date <= DATE '{D.date()}' - INTERVAL 30 DAY
      AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY
      AND f.report_date <= DATE '{D.date()}' + INTERVAL 30 DAY
    GROUP BY 1, 2
)
SELECT * FROM windowed
"""

label = con.sql(SQL).df()
print(f"content rows from the window: {len(label):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

content rows from the window: 333,275


In [6]:
MIN_PRIOR_IMP = 100   # min-volume floor: tiny pages are noise
MIN_OBS_DAYS = 7      # a real page shows up on several days, not one spike
DECLINE_FACTOR = 0.8  # future < 80% of prior counts as declined

label["labelable"] = (
    (label["prior_imp"] >= MIN_PRIOR_IMP)
    & (label["prior_obs_days"] >= MIN_OBS_DAYS)
    & (label["future_obs_days"] >= MIN_OBS_DAYS)
)
label["declined_30d_future"] = np.nan
label.loc[label["labelable"], "declined_30d_future"] = (
    label.loc[label["labelable"], "future_imp"]
    < DECLINE_FACTOR * label.loc[label["labelable"], "prior_imp"]
).astype(int)
label["declined_30d_future"] = label["declined_30d_future"].astype("Int64")

n_labelable = int(label["labelable"].sum())
print(f"content rows: {len(label):,} | labelable: {n_labelable:,} "
      f"| filtered out: {len(label) - n_labelable:,}")
print(f"base rate of declined_30d_future: {label['declined_30d_future'].mean():.3f}")
label.head()

content rows: 333,275 | labelable: 100,785 | filtered out: 232,490
base rate of declined_30d_future: 0.655


,client_hash_id,content_hash_id,prior_imp,prior_obs_days,future_imp,future_obs_days,labelable,declined_30d_future
0,client_23a62021009f63c4,content_b702bebc73a1287a,2112.0,30,921.0,30,True,1
1,client_23a62021009f63c4,content_b829695d26168a71,248.0,30,136.0,29,True,1
2,client_23a62021009f63c4,content_bbb23e9b149a90f2,237.0,30,345.0,30,True,0
3,client_23a62021009f63c4,content_bc1dc945322da28c,2486.0,30,1072.0,30,True,1
4,client_23a62021009f63c4,content_bc5f0720baffc3ab,1242.0,30,980.0,30,True,1


In [7]:
# Verification: one row per content, and the label window fully observed for every labeled page.
aligned = label[label["labelable"]]
print("one row per content:", label["content_hash_id"].is_unique)
print("future window observed days — min/median/max:",
      aligned["future_obs_days"].min(), int(aligned["future_obs_days"].median()),
      aligned["future_obs_days"].max())
print("prior  window observed days — min/median/max:",
      aligned["prior_obs_days"].min(), int(aligned["prior_obs_days"].median()),
      aligned["prior_obs_days"].max())
print(f"labelable clients: {aligned['client_hash_id'].nunique()}")

# Cache the small per-content label table for w05 (derived output, never a raw dataset).
cwd = Path.cwd()
OUT = cwd / "work" / "outputs" if (cwd / "work").is_dir() else cwd.parent / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
cache = label[["client_hash_id", "content_hash_id", "prior_imp", "prior_obs_days",
              "future_imp", "future_obs_days", "declined_30d_future"]]
cache.to_csv(OUT / "label_declined_30d.csv", index=False)
print("cached to:", OUT / "label_declined_30d.csv")

one row per content: True
future window observed days — min/median/max: 7 30 30
prior  window observed days — min/median/max: 7 30 30
labelable clients: 41


cached to: /Users/wyatt/Documents/programming/flyrank/work/outputs/label_declined_30d.csv


**Window alignment — the only overlap trap.** The label lives in June 2026, the final month of the panel. `fact_content_query_90d` covers a fixed 90-day window that *includes* those months, so its `impressions_90d` / `*_last30` columns CONTAIN the label period — using them as features would be leakage (`docs/data-dictionary.md:141`). This baseline builds features only from the daily fact table, with windows ending at `D`, so it has no overlap. If query-mix features are added later, only the `*_prev30` columns are safe.

Section 1 states the rule; section 2 scores and ranks this exact slice and writes `work/outputs/baseline_action_score.csv`.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
# Next: state the rule in plain words, then code it as a transparent score on the `label` table.
# Reason codes live here (e.g. "stale_visible_page", "declining_with_demand").
pass

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# Next: score every labelable row, rank descending, write work/outputs/baseline_action_score.csv.
# Evaluate precision@K against declined_30d_future and print the base rate (labels.mean()) next to it.
pass

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# Next: top-20 table — content, action, reason code, confidence, what would make it wrong.
pass

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# Next: name the weak picks; confirm no trend_direction/trend_pct, product flags, or future windows.
pass

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.